![logo_itq](img/logo-itq.jpeg)
## Detección de Outliers (Heart Disease)
*Nixon Malquin* — 24/05/2026

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../dataset/heart.csv')
X = df.drop(columns=['target'])
print('Forma:', X.shape)
X.head()

### 1) EllipticEnvelope (asume datos gaussianos)

In [ ]:
algorithm = EllipticEnvelope(support_fraction=None, contamination=0.1, random_state=42)
outlier_method = algorithm.fit(X)
labels = outlier_method.predict(X)
pos_out = np.where(labels == -1)[0]
print('Numero de outliers (EllipticEnvelope):', len(pos_out))

In [ ]:
# Funcion genérica como en clase
def find_outliers(df, algorithm):
    if algorithm.__class__.__name__ == 'LocalOutlierFactor':
        labels = algorithm.fit_predict(df)
    else:
        m = algorithm.fit(df)
        labels = m.predict(df)
    pos = np.where(labels == -1)[0]
    return labels, pos

### 2) Otros métodos: IsolationForest y LocalOutlierFactor

In [ ]:
IF  = IsolationForest(random_state=42)
LOF = LocalOutlierFactor(n_neighbors=10, metric='euclidean')

_, pos_if  = find_outliers(X, IF)
_, pos_lof = find_outliers(X, LOF)
print('Outliers IsolationForest    :', len(pos_if))
print('Outliers LocalOutlierFactor:', len(pos_lof))

### 3) Boxplot por atributo (regla del IQR x 1.5)

In [ ]:
def find_limits_BP(variable):
    Q1 = stats.scoreatpercentile(variable, 25)
    Q3 = stats.scoreatpercentile(variable, 75)
    RIC = Q3 - Q1
    li = Q1 - 1.5 * RIC
    ls = Q3 + 1.5 * RIC
    pos = np.where((variable < li) | (variable > ls))[0]
    return li, ls, pos

print(f"{'Atributo':<10} {'li':>8} {'ls':>8} {'#outliers':>10}")
for col in X.columns:
    li, ls, pos = find_limits_BP(X[col].values)
    print(f'{col:<10} {li:>8.2f} {ls:>8.2f} {len(pos):>10}')

plt.figure(figsize=(14, 5))
X.boxplot(rot=45)
plt.title('Boxplot por atributo (datos originales)')
plt.tight_layout()
plt.show()

### Interpretación
El boxplot detecta valores extremos en:
- `chol` (16 outliers) → un paciente con chol=564 (récord del dataset).
- `trestbps` (30 outliers) → presiones arteriales altas.
- `oldpeak` (7), `thalach` (4), `thal` (7).
- `fbs` (153) y `ca` (87): son técnicamente outliers según IQR, pero son **valores categóricos válidos** (fbs=1 cuando glucosa>120, ca=2/3/4 cuando hay vasos obstruidos), no errores de medición.

**Decisión:** *no eliminamos outliers*. En un contexto clínico, los valores extremos (colesterol muy alto, presión muy alta) son justamente los más informativos para detectar enfermedad. Eliminarlos quitaría señal útil al modelo. Sí los dejamos documentados.

### Link de repositorio
https://github.com/Ngmalquin123/MachingITQ